# Version 7 — Middleware: Clean Tool Output

We add a second middleware, `ToolOutputMiddleware`, that cleans `run_query` output before the model reads it. The budget middleware from Version 6 stays.

## Workflow

```text
 model asks for run_query(sql)
            |
            v
 SQLBudgetMiddleware        (outer — listed first)
   budget used up? ──yes──► "SQL budget reached ..." ──────────────┐
            | no                                                    |
            v                                                       |
 ToolOutputMiddleware       (inner — listed second)                 |
            |                                                       |
            v                                                       |
        run_query ──► raw text                                      |
            |                                                       |
            v                                                       |
   clean the text:                                                  |
     empty            ──► "TOOL OUTPUT VALIDATION: ... empty ..."   |
     SQL error/REFUSED ──► "SQL execution did not return business   |
                            data. " + first 1,200 chars             |
     > 5,000 chars    ──► cut + "[Output shortened ... do not       |
                            infer missing rows.]"                   |
            |                                                       |
            v                                                       |
 SQLBudgetMiddleware adds sql_call_count += 1                       |
            |                                                       |
            v                                                       v
                        result goes back to the model
```

The first middleware in the list is the outermost. So the budget check happens before the tool runs, and the output is cleaned before the budget adds its count.

## What changes from Version 6

| | Version 6 | Version 7 |
|---|---|---|
| Tools | 6 | same 6, unchanged |
| Prompt | Version 5 prompt | **unchanged** |
| Middleware | `SQLBudgetMiddleware` | `SQLBudgetMiddleware` **+** `ToolOutputMiddleware` |

**Why two classes:** one decides whether a call may run. The other changes what the call returns. Kept apart, each is easy to test and change.

**Why limit output:** a large result fills the model's context, costs tokens and can push the instructions out of focus. The marker tells the model the table is incomplete so it does not invent the missing rows.

## 1. Install dependencies

In [ ]:
%pip install -q langchain langchain-aws langchain-openai pinecone python-dotenv "psycopg[binary]" tavily-python

## 2. Configuration

| Variable | Required | Used for |
|---|---|---|
| `PINECONE_API_KEY` | yes | Reads the `ecom-kb` index |
| `PINECONE_INDEX_NAME` | no (default `ecom-kb`) | Index name |
| `PINECONE_EMBEDDING_MODEL` | no (default `text-embedding-3-small`) | Must match the model used to fill the index |
| `OPENAI_API_KEY` | yes | Query embeddings only |
| `BEDROCK_CHAT_MODEL_ID` | yes | The chat model (the brain) |
| `AWS_REGION` | no (default `us-east-1`) | Bedrock region; AWS credentials come from your normal AWS config |
| `PGHOST`, `PGDATABASE`, `PGUSER`, `PGPASSWORD` | yes | PostgreSQL connection (use a read-only role) |
| `PGPORT` | no (default `5432`) | |
| `PGSSLMODE` | no | e.g. `require` if your provider needs TLS |
| `TAVILY_API_KEY` | for web questions | Without it, `web_search` returns an error message |

Put these in a `.env` file next to the notebook. Never commit credentials.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Configuration — Version 7
#
#   .env ──► load_dotenv ──► checks ──► model        (Bedrock chat model: the brain)
#                                    ├─► embeddings   (OpenAI: turns a search phrase into a vector)
#                                    ├─► index        (Pinecone `ecom-kb`: the knowledge base)
#                                    └─► PostgreSQL settings (used by the SQL tools)
#                                    └─► tavily_client (web search, optional)
#
# The embedding model MUST be the same one used when the vectors were
# written to Pinecone, otherwise similarity scores are meaningless.
#
# Does NOT build or upload the knowledge base. It reads the existing index.
# ─────────────────────────────────────────────────────────────
import os
import json
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone
from tavily import TavilyClient

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "ecom-kb")
EMBEDDING_MODEL = os.getenv("PINECONE_EMBEDDING_MODEL", "text-embedding-3-small")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_CHAT_MODEL_ID = os.getenv("BEDROCK_CHAT_MODEL_ID", "")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY.")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY for query embeddings. It must match the embedding model used to build the Pinecone index.")
if not BEDROCK_CHAT_MODEL_ID:
    raise ValueError("Set BEDROCK_CHAT_MODEL_ID to a chat model enabled in your AWS account.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.getenv(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.getenv("PGPORT", "5432"))
PGSSLMODE = os.getenv("PGSSLMODE")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pinecone.Index(PINECONE_INDEX_NAME)
model = ChatBedrockConverse(
    model=BEDROCK_CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# Tavily is used only for external / public web information.
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "")
tavily_client = TavilyClient(api_key=TAVILY_API_KEY) if TAVILY_API_KEY else None
if tavily_client is None:
    print("TAVILY_API_KEY not set: web_search will return an error message instead of results.")

## 3. Carried over: all six tools

These cells are unchanged from Version 3, 4 and 5. Run them; the explanation is in Version 3, 4 and 5.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Knowledge-base tools — the agent's MEMORY
#
#   lookup_schema(term)        ──► search_namespace("nlq-schema")   ──► tables, columns, joins
#   lookup_examples(question)  ──► search_namespace("nlq-examples") ──► similar question + SQL
#   lookup_common(question)    ──► search_namespace("common")       ──► business definitions
#
#   search_namespace: text ──► embedding ──► Pinecone top-k ──► [{id, score, metadata}]
#   format_matches:   matches ──► readable text (no vectors), cut to max_chars
#
# Does NOT execute SQL and does NOT write to Pinecone.
# ─────────────────────────────────────────────────────────────


def search_namespace(namespace: str, text: str, top_k: int = 8) -> list[dict]:
    """Embed `text` and return the top-k matches from one Pinecone namespace."""
    vector = embeddings.embed_query(text)
    response = index.query(vector=vector, namespace=namespace, top_k=top_k, include_metadata=True)

    # The Pinecone client can return a dict or an object; read both shapes.
    matches = response.get("matches", []) if isinstance(response, dict) else response.matches
    results = []
    for match in matches:
        if isinstance(match, dict):
            results.append({"id": match.get("id", "unknown"),
                            "score": match.get("score"),
                            "metadata": match.get("metadata") or {}})
        else:
            results.append({"id": getattr(match, "id", "unknown"),
                            "score": getattr(match, "score", None),
                            "metadata": getattr(match, "metadata", None) or {}})
    return results


def format_matches(matches: list[dict], max_chars: int = 12000) -> str:
    """Turn matches into text blocks: 'RESULT n | id=... | score=...' plus metadata lines."""
    if not matches:
        return "No matching knowledge-base records were found."
    blocks = []
    for rank, match in enumerate(matches, start=1):
        lines = [f"RESULT {rank} | id={match['id']} | score={match['score']}"]
        for key, value in match["metadata"].items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            lines.append(f"{key}: {value}")
        blocks.append("\n".join(lines))
    output = "\n\n---\n\n".join(blocks)
    return output[:max_chars] + ("\n...[truncated]" if len(output) > max_chars else "")


@tool
def lookup_schema(term: str) -> str:
    """Search the nlq-schema Pinecone namespace for tables and columns.
    Use it to confirm real table/column names, data types, descriptions, synonyms,
    and business meaning. Retrieves metadata only; does not execute SQL."""
    return format_matches(search_namespace("nlq-schema", term, top_k=8))


@tool
def lookup_examples(question: str) -> str:
    """Search the nlq-examples Pinecone namespace for similar approved NL-to-SQL examples.
    Use examples as patterns, then verify schema details using lookup_schema.
    Retrieves examples only; does not execute SQL."""
    return format_matches(search_namespace("nlq-examples", question, top_k=4), max_chars=10000)


@tool
def lookup_common(question: str) -> str:
    """Search the common Pinecone namespace for shared business definitions or guidance.
    Retrieves knowledge only; does not execute SQL."""
    return format_matches(search_namespace("common", question, top_k=4), max_chars=8000)

In [ ]:
# ─────────────────────────────────────────────────────────────
# check_read_only_sql(sql) — ONE SQL policy, used by both SQL tools
#
#   sql ──► STEP 1 make a "scan copy": remove string literals, then comments
#       ──► STEP 2 exactly one statement?           (a ';' inside a string is fine)
#       ──► STEP 3 starts with SELECT or WITH?
#       ──► STEP 4 no write / admin keyword or risky function anywhere?
#       ──► (True, sql to run)  or  (False, reason)
#
# Why a scan copy: `WHERE status = 'delete requested'` must not be
# rejected because of the word inside the quotes.
#
# Does NOT talk to the database, and is NOT a complete SQL parser.
# The read-only database role is still the real boundary.
# ─────────────────────────────────────────────────────────────
import re

_FORBIDDEN = re.compile(
    r"(?is)\b("
    r"INSERT|UPDATE|DELETE|MERGE|CREATE|ALTER|DROP|TRUNCATE|GRANT|REVOKE|"
    r"CALL|COPY|VACUUM|REFRESH|LOCK|INTO|"          # INTO blocks SELECT ... INTO new_table
    r"pg_sleep|pg_read_file|pg_read_binary_file|pg_ls_dir|lo_import|lo_export|"
    r"dblink\w*|set_config|pg_terminate_backend|pg_cancel_backend"
    r")\b"
)


def _scan_copy(sql: str) -> str:
    sql = re.sub(r"'(?:[^']|'')*'", "''", sql)          # string literals -> ''
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)     # /* block comments */
    sql = re.sub(r"--[^\n]*", " ", sql)                  # -- line comments
    return sql.strip()


def check_read_only_sql(sql: str) -> tuple[bool, str]:
    """Return (True, sql_to_run) if the SQL passes the policy, else (False, reason)."""
    if not isinstance(sql, str) or not sql.strip():
        return False, "SQL must be a non-empty string."

    # STEP 1 — scan copy.
    scan = _scan_copy(sql)
    if not scan:
        return False, "SQL is empty after removing comments."

    # STEP 2 — one statement only (one trailing ';' allowed).
    scan = scan[:-1].rstrip() if scan.endswith(";") else scan
    if ";" in scan:
        return False, "Only one SQL statement is allowed."

    # STEP 3 — read-only statement types only.
    if not re.match(r"(?is)^(SELECT|WITH)\b", scan):
        return False, "Only read-only SELECT or WITH queries are allowed."

    # STEP 4 — no writes hidden anywhere (e.g. inside a CTE).
    found = _FORBIDDEN.search(scan)
    if found:
        return False, f"The SQL contains a write or administrative operation ({found.group(0).upper()}) and was rejected."

    # The SQL we run is the ORIGINAL text (literals intact), minus trailing ';'.
    return True, sql.strip().rstrip(";").strip()

In [ ]:
# ─────────────────────────────────────────────────────────────
# PostgreSQL helpers (shared by the SQL tools)
#
#   pg_connect()   ──► new connection, read-only by default, with a statement timeout
#   format_rows()  ──► columns + rows ──► plain-text table the model can read
#
# Does NOT replace a read-only database role. The role is the real
# security boundary; these settings are a second layer.
# ─────────────────────────────────────────────────────────────
MAX_ROWS = 50


def pg_connect(timeout_ms: int = 15000):
    """Open a PostgreSQL connection whose transactions are read-only."""
    kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
        "options": f"-c default_transaction_read_only=on -c statement_timeout={timeout_ms}",
    }
    if PGSSLMODE:
        kwargs["sslmode"] = PGSSLMODE
    return psycopg.connect(**kwargs)


def format_rows(columns: list[str], rows: list[dict], truncated: bool) -> str:
    """Render rows as 'col | col' lines. States clearly when rows were cut off."""
    if not rows:
        return "Query succeeded. 0 rows returned."
    header = " | ".join(columns)
    body = "\n".join(" | ".join(str(row.get(c)) for c in columns) for row in rows)
    tail = (f"({len(rows)} rows shown. More rows exist; the result was capped at {MAX_ROWS}.)"
            if truncated else f"({len(rows)} rows.)")
    return f"{header}\n{body}\n{tail}"

In [ ]:
# ─────────────────────────────────────────────────────────────
# validate_query(sql) — check the SQL BEFORE running it
#
#   sql ──► STEP 1 check_read_only_sql (policy, no database)
#       │        └─ fails ──► "VALIDATION FAILED: <reason>"
#       ──► STEP 2 PostgreSQL: EXPLAIN <sql>   (parses + plans, does not run it)
#       │        └─ fails ──► "VALIDATION FAILED: PostgreSQL could not validate ...: <error>"
#       ──► "VALIDATION PASSED ... You may now call run_query(sql)." + plan preview
#
# EXPLAIN catches unknown tables/columns, type errors and syntax errors
# without touching any business data.
#
# Does NOT execute the query and does NOT return business rows.
# ─────────────────────────────────────────────────────────────
@tool
def validate_query(sql: str) -> str:
    """Validate a generated SQL query before execution.
    Checks that it is a single read-only SELECT/WITH statement and asks PostgreSQL
    EXPLAIN to parse and plan it without running the query. Call this before run_query.
    This tool does not execute the query or return business data."""
    # STEP 1 — policy.
    ok, sql_or_reason = check_read_only_sql(sql)
    if not ok:
        return f"VALIDATION FAILED: {sql_or_reason}"

    # STEP 2 — let PostgreSQL parse and plan it.
    try:
        with pg_connect(timeout_ms=10000) as conn:
            plan = conn.execute("EXPLAIN " + sql_or_reason).fetchall()
    except Exception as exc:
        # Short, actionable message; no connection details.
        message = str(exc).splitlines()[0][:500]
        return f"VALIDATION FAILED: PostgreSQL could not validate the query: {message}"

    plan_text = "\n".join(str(row.get("QUERY PLAN", row)) for row in plan[:8])
    return (
        "VALIDATION PASSED: PostgreSQL accepted the query for planning. "
        "The query has not been executed. You may now call run_query(sql) with this exact SQL.\n"
        f"Query plan preview:\n{plan_text[:3000]}"
    )

In [ ]:
# ─────────────────────────────────────────────────────────────
# run_query(sql) — the agent's HANDS (now uses the shared SQL policy)
#
#   sql ──► STEP 1 check_read_only_sql   (same rules as validate_query)
#       ──► STEP 2 execute on a read-only connection
#       ──► STEP 3 format up to 50 rows as text ──► back to the model
#
# Why check again here: the prompt TELLS the model to validate first,
# but a prompt is not enforcement. If the model skips validate_query,
# run_query still refuses unsafe SQL.
#
# Does NOT run EXPLAIN; that is validate_query's job.
# ─────────────────────────────────────────────────────────────
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SELECT or WITH query against PostgreSQL.
    Call validate_query first. Returns at most 50 rows as text, or a message
    starting with "SQL error:" or "REFUSED:" that explains what went wrong."""
    # STEP 1 — policy.
    ok, sql_or_reason = check_read_only_sql(sql)
    if not ok:
        return f"REFUSED: {sql_or_reason}"

    # STEP 2 — execute. Fetch one extra row so we know if the result was cut off.
    try:
        with pg_connect() as conn:
            cursor = conn.execute(sql_or_reason)
            fetched = cursor.fetchmany(MAX_ROWS + 1)
            columns = [d.name for d in cursor.description] if cursor.description else []
    except psycopg.Error as exc:
        return f"SQL error: {exc}"

    # STEP 3 — format for the model.
    return format_rows(columns, fetched[:MAX_ROWS], truncated=len(fetched) > MAX_ROWS)

In [ ]:
# ─────────────────────────────────────────────────────────────
# web_search(query, include_domains, exclude_domains, max_results)
#
#   query ──► STEP 1 checks (non-empty, Tavily configured)
#         ──► STEP 2 clean domains: "https://www.oecd.org/x" -> "www.oecd.org"
#         ──► STEP 3 Tavily search (basic depth, 1–10 results)
#         ──► dict with results (title, url, content) or {"error": ...}
#
# Does NOT query internal data. Web results are external evidence,
# never facts about the marketplace's own business.
# ─────────────────────────────────────────────────────────────
from typing import Any


def _normalize_domains(domains: list[str] | None) -> list[str] | None:
    if not domains:
        return None
    cleaned = []
    for domain in domains:
        if not domain or not domain.strip():
            continue
        value = domain.strip().lower().removeprefix("https://").removeprefix("http://").strip("/")
        cleaned.append(value.split("/", 1)[0])
    return cleaned or None


@tool
def web_search(
    query: str,
    include_domains: list[str] | None = None,
    exclude_domains: list[str] | None = None,
    max_results: int = 5,
) -> dict[str, Any]:
    """Search the public web for current or external information using Tavily.

    Args:
        query: What information to search for.
        include_domains: Optional allowed domains, e.g. ["nist.gov", "oecd.org"].
        exclude_domains: Optional domains to exclude, e.g. ["reddit.com"].
        max_results: Number of results to return, clamped to 1-10.

    Use this tool when the question needs current external facts, industry research,
    public reports, or competitor information. Use domain filters when requested or
    when a domain restriction is useful. This tool does not query internal databases.
    """
    # STEP 1 — checks.
    if not query or not query.strip():
        return {"error": "Please provide a non-empty search query."}
    if tavily_client is None:
        return {"error": "Tavily is not configured. Set TAVILY_API_KEY in the environment."}

    # STEP 2 — build the request.
    kwargs: dict[str, Any] = {
        "query": query.strip(),
        "max_results": max(1, min(int(max_results), 10)),
        "search_depth": "basic",
    }
    include = _normalize_domains(include_domains)
    exclude = _normalize_domains(exclude_domains)
    if include:
        kwargs["include_domains"] = include
    if exclude:
        kwargs["exclude_domains"] = exclude

    # STEP 3 — search.
    try:
        return tavily_client.search(**kwargs)
    except Exception as exc:
        return {"error": f"Tavily search failed: {type(exc).__name__}: {str(exc)[:300]}"}

## 4. Carried over: `SQLBudgetMiddleware`

These cells are unchanged from Version 6. Run them; the explanation is in Version 6.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SQLBudgetMiddleware — at most N run_query executions per agent run
#
#   model asks for a tool
#        │
#        ▼
#   wrap_tool_call(request, handler)
#        │
#        ├─ STEP 1 not run_query?            ──► run the tool, count nothing
#        ├─ STEP 2 sql_call_count >= N?      ──► do NOT run the tool;
#        │                                       return "SQL budget reached ..." to the model
#        └─ STEP 3 otherwise                 ──► run the tool, then add 1 to sql_call_count
#                                                (a call that returns a SQL error still counts)
#
# The counter lives in AGENT STATE (`sql_call_count`). Its reducer is
# operator.add, so each update of 1 is added to the running total.
#
# Both hooks are defined:
#   wrap_tool_call   → used by agent.invoke()  / agent.stream()
#   awrap_tool_call  → used by agent.ainvoke() / agent.astream()
# Defining only the async one makes agent.invoke() raise NotImplementedError.
#
# Does NOT:
#   - limit lookups, validation or web search (they never touch the counter)
#   - reset between turns if you add a checkpointer + thread_id
#     (state persists, so the budget then covers the whole conversation)
#   - serialize parallel calls: several run_query calls requested in the SAME
#     model turn all see the same starting count
# ─────────────────────────────────────────────────────────────
import operator
from typing import Annotated
from typing_extensions import NotRequired

from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage
from langgraph.types import Command

SQL_TOOL_NAME = "run_query"
MAX_SQL_CALLS = 3


class SQLBudgetState(AgentState):
    # LangGraph adds each update to the existing value (0 if missing).
    sql_call_count: NotRequired[Annotated[int, operator.add]]


class SQLBudgetMiddleware(AgentMiddleware):
    """Limit calls to the run_query SQL execution tool."""

    state_schema = SQLBudgetState

    def __init__(self, max_sql_calls: int = MAX_SQL_CALLS):
        self.max_sql_calls = max_sql_calls

    # ---- shared logic -------------------------------------------------
    def _is_sql_call(self, request) -> bool:
        return request.tool_call.get("name", "") == SQL_TOOL_NAME

    def _budget_used_up(self, request) -> bool:
        return request.state.get("sql_call_count", 0) >= self.max_sql_calls

    def _budget_message(self, request) -> ToolMessage:
        return ToolMessage(
            content=(
                f"SQL budget reached: {self.max_sql_calls} run_query calls have already been used. "
                "Do not execute another SQL query. Answer using the results you already have, "
                "or explain what remains unknown."
            ),
            tool_call_id=request.tool_call["id"],
            name=SQL_TOOL_NAME,
        )

    @staticmethod
    def _add_one(result):
        """Attach `sql_call_count: +1` to whatever the tool returned."""
        if isinstance(result, ToolMessage):
            return Command(update={"messages": [result], "sql_call_count": 1})
        if isinstance(result, Command):
            update = dict(result.update or {})
            update["sql_call_count"] = update.get("sql_call_count", 0) + 1
            return Command(graph=result.graph, update=update, resume=result.resume, goto=result.goto)
        return result

    # ---- sync hook (agent.invoke) -------------------------------------
    def wrap_tool_call(self, request, handler):
        if not self._is_sql_call(request):          # STEP 1
            return handler(request)
        if self._budget_used_up(request):           # STEP 2
            return self._budget_message(request)
        return self._add_one(handler(request))      # STEP 3

    # ---- async hook (agent.ainvoke) -----------------------------------
    async def awrap_tool_call(self, request, handler):
        if not self._is_sql_call(request):          # STEP 1
            return await handler(request)
        if self._budget_used_up(request):           # STEP 2
            return self._budget_message(request)
        return self._add_one(await handler(request))  # STEP 3

## 5. New: `ToolOutputMiddleware`

The text rules live in one method, `clean()`, so we can test them without an agent. The middleware handles both shapes a tool result can take: a plain `ToolMessage`, or a `Command` that carries one.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ToolOutputMiddleware — clean run_query output before the model reads it
#
#   run_query result
#        │
#        ▼
#   STEP 1 find the ToolMessage (plain, or inside a Command update)
#   STEP 2 clean its text:
#            empty / not text        ──► "TOOL OUTPUT VALIDATION: run_query returned an empty or non-text result."
#            starts "SQL error:" / "REFUSED:" ──► prefix "SQL execution did not return business data."
#                                               and keep the first 1,200 chars of the error
#            longer than max chars   ──► cut, then add "[Output shortened ... do not infer missing rows.]"
#            otherwise               ──► unchanged
#   STEP 3 return the same kind of object with the cleaned text
#
# Does NOT:
#   - touch other tools' output (lookups, validation, web search)
#   - decide whether a call is allowed; that is SQLBudgetMiddleware's job
#   - change what is stored anywhere else in state
# ─────────────────────────────────────────────────────────────
MAX_TOOL_OUTPUT_CHARS = 5_000


class ToolOutputMiddleware(AgentMiddleware):
    """Validate, clarify, and limit run_query output before it reaches the model."""

    def __init__(self, max_output_chars: int = MAX_TOOL_OUTPUT_CHARS):
        self.max_output_chars = max_output_chars

    # ---- shared logic -------------------------------------------------
    def clean(self, content) -> str:
        """STEP 2 — the text rules. Public so it can be tested without an agent."""
        if not isinstance(content, str) or not content.strip():
            return "TOOL OUTPUT VALIDATION: run_query returned an empty or non-text result."
        if content.lstrip().lower().startswith(("sql error:", "refused:")):
            return "SQL execution did not return business data. " + content[:1200]
        if len(content) > self.max_output_chars:
            return (
                content[: self.max_output_chars]
                + f"\n\n[Output shortened to {self.max_output_chars} characters. "
                + "The displayed result may be incomplete; do not infer missing rows.]"
            )
        return content

    def _process(self, request, result):
        if request.tool_call.get("name", "") != SQL_TOOL_NAME:
            return result

        # STEP 1 + 3 — plain ToolMessage.
        if isinstance(result, ToolMessage):
            return result.model_copy(update={"content": self.clean(result.content)})

        # STEP 1 + 3 — Command whose update carries the ToolMessage.
        if isinstance(result, Command) and isinstance(result.update, dict):
            messages = result.update.get("messages") or []
            cleaned = [
                m.model_copy(update={"content": self.clean(m.content)}) if isinstance(m, ToolMessage) else m
                for m in messages
            ]
            update = {**result.update, "messages": cleaned}
            return Command(graph=result.graph, update=update, resume=result.resume, goto=result.goto)

        return result

    # ---- sync hook (agent.invoke) -------------------------------------
    def wrap_tool_call(self, request, handler):
        return self._process(request, handler(request))

    # ---- async hook (agent.ainvoke) -----------------------------------
    async def awrap_tool_call(self, request, handler):
        return self._process(request, await handler(request))

In [ ]:
# Test the cleaning rules offline — no agent, no database.
cleaner = ToolOutputMiddleware(max_output_chars=200)

print(cleaner.clean(""))
print("---")
print(cleaner.clean('SQL error: column "revnue" does not exist'))
print("---")
long_table = "product_id | revenue\n" + "\n".join(f"{i} | {i * 10.5}" for i in range(100))
print(cleaner.clean(long_table))

## 6. The system prompt — unchanged from Version 5

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace. Answer questions using
internal business data, the internal knowledge base, and public web sources when needed.

AVAILABLE TOOLS
1. lookup_schema(term): searches the Pinecone `nlq-schema` namespace for table and column metadata.
2. lookup_examples(question): searches the Pinecone `nlq-examples` namespace for similar questions and SQL examples.
3. lookup_common(question): searches the Pinecone `common` namespace for shared business definitions and guidance.
4. validate_query(sql): checks the read-only SQL policy and asks PostgreSQL to validate the query using EXPLAIN. It does not execute the business query.
5. run_query(sql): executes a read-only SQL query against PostgreSQL and returns internal business data.
6. web_search(query, include_domains, exclude_domains, max_results): searches the public web using Tavily. Domain filters are optional.

TOOL-SELECTION RULES
- Use Pinecone tools when internal schema, SQL examples, or business definitions are needed.
- Use PostgreSQL for questions about the marketplace's actual data, metrics, customers, products, orders, or transactions.
- Use web_search when the question requires current or external information, such as public market trends, competitor information, public reports, or industry benchmarks.
- Use both PostgreSQL and web_search when the user asks to compare internal performance with external information.
- Do not use web search as a substitute for querying internal business data.
- Do not call every tool automatically; select only the tools needed to answer the question.
- When the user specifies allowed websites, pass them through include_domains. When the user asks to exclude websites, pass them through exclude_domains. Do not invent arbitrary domain restrictions.
- Treat web results as external evidence, not as facts about the user's internal business.

SQL WORKFLOW
- Understand the business question, required metric, filters, and expected level of detail.
- Use lookup_examples when a similar approved SQL example may help.
- Use lookup_schema to confirm table names, columns, relationships, metric definitions, and table grain.
- Use lookup_common when shared business definitions or guidance may help.
- Treat retrieved SQL examples as patterns; do not copy mismatched joins, filters, date ranges, or columns.
- Never invent table names, columns, joins, value codes, or business definitions.
- Before executing SQL, call validate_query(sql).
- If validation fails, correct the SQL using the returned error and validate it again.
- Call run_query(sql) only after validation succeeds.
- Inspect the query result before answering. If SQL fails, use the error and schema knowledge to correct it when possible.
- Never claim that a query succeeded unless run_query confirms it.

EVIDENCE AND SAFETY
- Never invent numbers, query results, web findings, citations, or URLs.
- Use read-only SELECT/WITH queries only and respect the validation tool's restrictions.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state the numerator and denominator.
- For web-based claims, cite or list the source URLs returned by Tavily and distinguish facts from interpretation.
- When combining internal and external evidence, label which information came from PostgreSQL and which came from web sources.
- If the available evidence is insufficient or sources conflict, state the uncertainty clearly.
- Do not expose private chain-of-thought; provide a concise explanation of the method and evidence.

OUTPUT FORMAT
Use the sections relevant to the question. Do not force SQL-specific sections onto a web-only answer.

ANSWER:
Provide the answer with units where applicable, or state UNKNOWN if the evidence is insufficient.

DENOMINATOR:
For percentages, explain the numerator and denominator. For non-percentage answers, write N/A.

SOURCE:
For database answers, include the SQL query actually executed and identify PostgreSQL as the source. If no query was executed, say so. For web answers, list the source titles and URLs returned by Tavily. For combined answers, separate internal database sources from external web sources.

CAVEAT:
Mention limitations, ambiguity, time-period mismatches, data-quality concerns, or source uncertainty. If none are apparent, write "None identified from the available evidence."
"""

## 7. Create the agent with both middleware

In [ ]:
agent = create_agent(
    model=model,
    tools=[lookup_schema, lookup_examples, lookup_common, validate_query, run_query, web_search],
    system_prompt=SYSTEM_PROMPT,
    middleware=[
        SQLBudgetMiddleware(max_sql_calls=MAX_SQL_CALLS),        # outer
        ToolOutputMiddleware(max_output_chars=MAX_TOOL_OUTPUT_CHARS),  # inner
    ],
)

## 8. Run the agent

The second question asks for a long listing. Look for the **[Output shortened …]** marker in the trace, and check that the answer does not invent rows beyond it.

In [ ]:
# ─────────────────────────────────────────────────────────────
# ask_agent(question) — run the agent once and print what happened
#
#   question ──► agent.invoke ──► final state
#                                   │
#                                   ├─ AIMessage with tool_calls ──► "BRAIN → TOOL"
#                                   ├─ ToolMessage               ──► "TOOL → OBSERVATION"
#                                   └─ last message              ──► "FINAL ANSWER"
#
# Does NOT change the agent. It only reads the messages it returns.
# ─────────────────────────────────────────────────────────────
def ask_agent(question: str, agent_to_use=None, max_obs_chars: int = 1500):
    agent_to_use = agent_to_use or agent
    result = agent_to_use.invoke({"messages": [{"role": "user", "content": question}]})

    line = "=" * 80
    print(f"{line}\nQUESTION\n{line}\n{question}")
    print(f"\n{line}\nAGENT TRACE\n{line}")

    for message in result["messages"]:
        kind = type(message).__name__
        if kind == "AIMessage" and getattr(message, "tool_calls", None):
            for call in message.tool_calls:
                print("\n🧠 BRAIN → TOOL")
                print("Tool:", call["name"])
                print("Arguments:", call["args"])
        elif kind == "ToolMessage":
            print(f"\n🖐️ TOOL → OBSERVATION ({message.name})")
            print(str(message.content)[:max_obs_chars])

    print(f"\n{line}\nFINAL ANSWER\n{line}")
    print(result["messages"][-1].content)
    if "sql_call_count" in result:
        print("\nrun_query executions used:", result["sql_call_count"])
    return result

In [ ]:
result = ask_agent("What are the top 5 products by revenue?")

In [ ]:
result = ask_agent("List every order from last month with its customer and total.", max_obs_chars=6000)

## 9. Key points

- Middleware order matters: the first in the list wraps all the others.
- One middleware decides **if** a call runs; another shapes **what** it returns.
- Put the text rules in a plain method so they can be tested offline.
- Handle both `ToolMessage` and `Command` results.

**Still free text:** the final answer is prose. A supervisor or chart agent cannot reliably read columns and rows out of prose.

**Next — Version 8:** structured output. The model returns a small typed decision, and Python builds the final table from what `run_query` actually returned.